## Loading functions from other jupyter notebooks

#### Jupyter Notebooks are techinically saved as JSON files and not Python scripts (.py files), standard Python "import" command does not work here.

In [1]:
%run 01_pdf_splitting.ipynb
%run 02_spacy_deontic.ipynb
%run 03_reference_linker.ipynb

--- Chunk 1 ---
1. The User may use the Service on the premise that they agree to these Terms and Conditions. The User shall use the Service within the scope specified by us according to the conditions stipulated by us, such as usage and usage environment. When using the Service, the User shall always confirm these Terms and Conditions for each function or service.

--- Chunk 2 ---
2. If necessary, we may add new services to the Service, change the contents or name of the Service, and the User shall use the Service upon prior consent to these changes. Article 2 (User Information)

--- Chunk 3 ---
1. We may obtain the following information (hereinafter referred to as “User Information") regarding the Service. (1) Personal information: The User's name, email address, date of birth and gender. (2) Information about the terminal used: Device, OS, the User agent, cookies, advertisement identifier "IDFA/Android Ad ID", terminal language setting, access country, etc. (3) Information on commun

#### Or you could put those functions in a Python script and then import those functions here. (It is considered the best practice to follow.)

In [2]:
# Importing our custom functions directly from extractors.py
from ingestion.parser import split_into_clauses, extract_deontic_metadata, extract_cross_references

ModuleNotFoundError: No module named 'ingestion'

#### the above, runs the notebooks and imports their functions into this one

In [ ]:
def process_legal_clause(clause_text, parent_heading="General"):
    """
    Takes a raw clause, enriches it, and extracts all legal metadata
    for vector database indexing.
    """
    # 1. Contextual Enrichment (Adding the parent scope)
    enriched_text = f"[{parent_heading}] {clause_text}"
    
    # 2. Extract Deontic Logic
    deontic_tags = extract_deontic_metadata(clause_text)
    
    # 3. Extract 1-Hop References
    links = extract_cross_references(clause_text)
    
    # 4. Build the final Payload
    payload = {
        "id": hash(clause_text), # Creates a unique ID for the database
        "enriched_text": enriched_text,
        "metadata": {
            "parent_scope": parent_heading,
            "obligations": ", ".join(deontic_tags["Obligations"]),
            "permissions": ", ".join(deontic_tags["Permissions"]),
            "prohibitions": ", ".join(deontic_tags["Prohibitions"]),
            "conditions": ", ".join(deontic_tags["Conditions"]),
            "linked_nodes": ", ".join(links)
        }
    }
    
    return payload

## TESTING IT OUT

In [ ]:
abema_sample_clause = """
Article 5. Usage Fees
The User shall pay the usage fee calculated by the Company, subject to Section 8.1. 
The User may cancel the subscription at any time.
"""

# Processing the clause
processed_data = process_legal_clause(abema_sample_clause, parent_heading="Article 5: Usage Fees")

# Printing the structured output
import json
print(json.dumps(processed_data, indent=4))

{
    "id": -3966153002339510735,
    "enriched_text": "[Article 5: Usage Fees] \nArticle 5. Usage Fees\nThe User shall pay the usage fee calculated by the Company, subject to Section 8.1. \nThe User may cancel the subscription at any time.\n",
    "metadata": {
        "parent_scope": "Article 5: Usage Fees",
        "obligations": "shall pay the usage fee calculated",
        "permissions": "may cancel the subscription at any",
        "prohibitions": "",
        "conditions": "subject to Section 8.1. \n",
        "linked_nodes": "Linked Node: 8.1"
    }
}
